# LangChain Price-Comparison Agent — Step-by-Step Build

This notebook walks through the full build from the `PLAN.md`:
- **Run A** — unmonitored baseline (custom execution-trace logger)
- **Run B** — AgentTrust-wrapped (per-step governance + final-output validation)
- **Analysis** — join the two traces on `correlation_id` and compare

Work through each section sequentially. Each step produces verifiable output before you move to the next.

## Step 0 — Environment Setup

Install all required packages. Run this once; restart the kernel after if prompted.

In [2]:
import subprocess, sys

# Pinned to the 0.3.x line on purpose: langchain 1.x reorganized/removed several
# APIs this notebook relies on (StructuredTool, langchain.callbacks.base,
# create_react_agent, langchain_core.prompts). Pinning here means Step 0 only
# needs to run once — no more chasing ImportErrors cell-by-cell after each
# kernel restart. Re-running this cell after packages are already installed is
# a fast no-op, so restarting the kernel is cheap.
#
# Note: langchain-openai is no longer needed — the agent's LLM backbone is
# now the local `claude` CLI (agent/claude_code_llm.py), not the OpenAI API.
packages = [
    "langchain==0.3.9",
    "langchain-community==0.3.9",
    "requests",
    "beautifulsoup4",
    "python-dotenv",
    "pandas",
    "matplotlib",
]

for pkg in packages:
   subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-q", pkg])

print("✓ Core packages installed")

# AgentTrust SDK — adjust the path to match your local clone
# Uncomment whichever line applies:
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/path/to/agentrust_sdk"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/path/to/agentrust_sdk[embedded,retry]"])
print("⚠  AgentTrust install line is commented out — edit the path above and uncomment")

✓ Core packages installed
⚠  AgentTrust install line is commented out — edit the path above and uncomment


In [ ]:
import os
from pathlib import Path

# Create the directory layout used throughout this notebook
for d in ["agent", "runs", "analysis", "policy", "logs"]:
    Path(d).mkdir(exist_ok=True)

print("Directory structure ready:")
for d in ["agent", "runs", "analysis", "policy", "logs"]:
    print(f"  ./{d}/")

## Step 0b — Environment Variables

Write your `.env` file. Fill in the real keys before running Step 1.

In [ ]:
from pathlib import Path

env_content = """\
# LLM backbone: this notebook now uses your local `claude` CLI session
# (agent/claude_code_llm.py) instead of the OpenAI API, so no
# OPENAI_API_KEY / ANTHROPIC_API_KEY is required here. Just make sure
# `claude auth status` shows loggedIn: true before running Step 5.

# AgentTrust
AGENTRUST_KEY=at_...
AGENTRUST_GATEWAY_URL=http://localhost:8765
AGENTRUST_FAILURE_MODE=open
AGENTRUST_ENABLED=true
"""

env_path = Path(".env")
if not env_path.exists():
    env_path.write_text(env_content)
    print(".env created — fill in your real keys")
else:
    print(".env already exists — skipping overwrite")

# Load into the current kernel session
from dotenv import load_dotenv
load_dotenv(override=True)

import subprocess as _sp
_claude_status = _sp.run(["claude", "auth", "status"], capture_output=True, text=True)
print("claude CLI auth status:")
print(_claude_status.stdout.strip() or _claude_status.stderr.strip())

## Step 1 — Define the Tools

Three standalone functions — test each one **before** wiring into LangChain.

| Tool | What it does |
|---|---|
| `fetch_html` | GET an approved URL, return HTML + timing |
| `extract_price` | Parse a `$XX.XX` price out of HTML |
| `compare_prices` | Return the cheaper site + dollar difference |

In [ ]:
%%writefile agent/tools.py
import re
import time
import requests
from bs4 import BeautifulSoup

# Replace with the two real product-page domains you want to compare
APPROVED_DOMAINS = ["site-a.example.com", "site-b.example.com"]


def fetch_html(url: str) -> dict:
    """Fetch raw HTML for an approved domain. Returns status, size, timing, html."""
    domain = url.split("/")[2]
    if domain not in APPROVED_DOMAINS:
        raise ValueError(f"Domain not approved: {domain}")

    start = time.time()
    resp = requests.get(url, timeout=10)
    elapsed = time.time() - start

    return {
        "url": url,
        "status": resp.status_code,
        "html": resp.text,
        "size": len(resp.text),
        "elapsed_sec": round(elapsed, 3),
    }


def extract_price(html: str) -> dict:
    """Parse a price out of HTML. Returns value + match count for validation."""
    soup = BeautifulSoup(html, "html.parser")
    matches = re.findall(r"\$\s?(\d+\.\d{2})", soup.get_text())
    return {
        "matches_found": len(matches),
        "price": float(matches[0]) if len(matches) == 1 else None,
        "raw_matches": matches,
    }


def compare_prices(price_a: float, price_b: float) -> dict:
    if price_a is None or price_b is None:
        return {"error": "Missing price — cannot compare"}
    cheaper = "A" if price_a < price_b else "B"
    return {
        "cheaper_site": cheaper,
        "price_a": price_a,
        "price_b": price_b,
        "difference": round(abs(price_a - price_b), 2),
    }

Overwriting agent/tools.py


### 1b — Smoke-test the tools with mock data

We use `unittest.mock` so no real HTTP is needed. Confirm all three tools return sane output before touching LangChain.

In [6]:
import importlib, sys
from unittest.mock import patch, MagicMock

# Reload in case cell 1 was edited
if "agent.tools" in sys.modules:
    del sys.modules["agent.tools"]

from agent.tools import fetch_html, extract_price, compare_prices

# --- fetch_html mock ---
mock_resp = MagicMock()
mock_resp.status_code = 200
mock_resp.text = '<html><body><span class="price">$29.99</span></body></html>'

with patch("agent.tools.requests.get", return_value=mock_resp):
    result_fetch = fetch_html("http://site-a.example.com/product")

print("fetch_html →", {k: v for k, v in result_fetch.items() if k != "html"})
assert result_fetch["status"] == 200, "Expected 200"

# --- extract_price ---
result_extract = extract_price(result_fetch["html"])
print("extract_price →", result_extract)
assert result_extract["price"] == 29.99, f"Expected 29.99, got {result_extract['price']}"

# --- compare_prices ---
result_compare = compare_prices(29.99, 34.50)
print("compare_prices →", result_compare)
assert result_compare["cheaper_site"] == "A"
assert result_compare["difference"] == 4.51

print("\n✓ All three tools pass smoke tests")

fetch_html → {'url': 'http://site-a.example.com/product', 'status': 200, 'size': 59, 'elapsed_sec': 0.0}
extract_price → {'matches_found': 1, 'price': 29.99, 'raw_matches': ['29.99']}
compare_prices → {'cheaper_site': 'A', 'price_a': 29.99, 'price_b': 34.5, 'difference': 4.51}

✓ All three tools pass smoke tests


## Step 2 — Wrap as LangChain `StructuredTool`s

Each tool gets a Pydantic input schema so LangChain can validate arguments before calling the function.

In [7]:
%%writefile agent/tools_wrapped.py
import json

from langchain.tools import StructuredTool
from pydantic import BaseModel
from agent.tools import fetch_html, extract_price, compare_prices


class FetchInput(BaseModel):
    url: str


class ExtractInput(BaseModel):
    html: str


class CompareInput(BaseModel):
    input: str


def _compare_prices_from_text(input: str) -> dict:
    """Adapter for the single-input ReAct agent, which can only pass one
    string per action (it can't map a raw JSON blob onto a 2-field schema).
    Accepts either a JSON object string or a "price_a, price_b" pair."""
    try:
        data = json.loads(input)
        price_a, price_b = data["price_a"], data["price_b"]
    except (json.JSONDecodeError, KeyError, TypeError):
        parts = [p.strip().strip("$") for p in input.replace(",", " ").split()]
        price_a, price_b = parts[0], parts[1]
    return compare_prices(float(price_a), float(price_b))


fetch_tool = StructuredTool.from_function(
    func=fetch_html,
    name="fetch_html",
    description="Fetch HTML from an approved product page URL.",
    args_schema=FetchInput,
)

extract_tool = StructuredTool.from_function(
    func=extract_price,
    name="extract_price",
    description="Extract a numeric price from fetched HTML.",
    args_schema=ExtractInput,
)

compare_tool = StructuredTool.from_function(
    func=_compare_prices_from_text,
    name="compare_prices",
    description=(
        "Compare two prices and return the cheaper option. "
        'Input must be a single JSON object string, e.g. {"price_a": 49.99, "price_b": 44.49}'
    ),
    args_schema=CompareInput,
)

Overwriting agent/tools_wrapped.py


### 2b — Verify tool schemas serialise correctly

In [2]:
import json, sys

# Force reload
for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.tools_wrapped import fetch_tool, extract_tool, compare_tool

tools = [fetch_tool, extract_tool, compare_tool]
for t in tools:
    schema = t.args_schema.model_json_schema()
    print(f"{t.name:20s} → {json.dumps(schema['properties'])}")

print("\n✓ All tool schemas look good")

fetch_html           → {"url": {"title": "Url", "type": "string"}}
extract_price        → {"html": {"title": "Html", "type": "string"}}
compare_prices       → {"price_a": {"title": "Price A", "type": "number"}, "price_b": {"title": "Price B", "type": "number"}}

✓ All tool schemas look good


## Step 3 — Custom Execution-Trace Callback Handler

The handler writes a JSONL file with one entry per tool start/end and agent action.  
The `correlation_id` generated at `tool_start` is the **join key** between this trace and the AgentTrust governance trace written in Step 7.

In [3]:
%%writefile agent/callback_handler.py
import json
import time
import uuid
from collections import defaultdict
from langchain.callbacks.base import BaseCallbackHandler


class ExecutionTraceHandler(BaseCallbackHandler):
    """Writes a JSONL execution trace.  Every tool-start/end pair shares a
    correlation_id so the governance trace can be joined on that key."""

    def __init__(self, run_id: str, log_path: str = "logs/execution_trace.jsonl"):
        self.run_id = run_id
        self.log_path = log_path
        self._pending: dict[int, str] = {}
        self._call_counter = defaultdict(int)

    def correlation_id_for(self, tool_name: str) -> str:
        key = self._call_counter[tool_name] - 1
        return self._pending.get(key, str(uuid.uuid4()))

    def _write(self, record: dict):
        record["run_id"] = self.run_id
        record["timestamp"] = time.time()
        with open(self.log_path, "a") as f:
            f.write(json.dumps(record) + "\n")

    def on_tool_start(self, serialized, input_str, **kwargs):
        cid = str(uuid.uuid4())
        serial = self._call_counter[serialized.get("name", "")]
        self._call_counter[serialized.get("name", "")] += 1
        self._pending[serial] = cid
        self._write({
            "event": "tool_start",
            "tool": serialized.get("name"),
            "input": input_str,
            "correlation_id": cid,
        })

    def on_tool_end(self, output, **kwargs):
        tool_name = kwargs.get("name", list(self._call_counter.keys())[-1] if self._call_counter else "unknown")
        serial = self._call_counter[tool_name] - 1
        cid = self._pending.get(serial, str(uuid.uuid4()))
        self._write({
            "event": "tool_end",
            "tool": tool_name,
            "output": str(output),
            "correlation_id": cid,
        })

    def on_agent_action(self, action, **kwargs):
        self._write({
            "event": "agent_action",
            "tool": action.tool,
            "tool_input": action.tool_input,
            "log": action.log,
            "correlation_id": str(uuid.uuid4()),
        })

Overwriting agent/callback_handler.py


### 3b — Verify the handler writes `correlation_id` in paired start/end rows

In [4]:
import uuid, json, os, sys
from pathlib import Path

Path("logs").mkdir(exist_ok=True)

# Clear any previous test log
test_log = Path("logs/test_handler.jsonl")
test_log.unlink(missing_ok=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.callback_handler import ExecutionTraceHandler

run_id = str(uuid.uuid4())
handler = ExecutionTraceHandler(run_id=run_id, log_path=str(test_log))

# Simulate tool_start → tool_end
handler.on_tool_start({"name": "fetch_html"}, '{"url": "http://site-a.example.com/product"}')
handler.on_tool_end('{"status": 200}', name="fetch_html")

records = [json.loads(l) for l in test_log.read_text().splitlines()]
print(f"Records written: {len(records)}")
for r in records:
    print(f"  event={r['event']:12s}  correlation_id={r['correlation_id']}")

start_cid = records[0]["correlation_id"]
end_cid   = records[1]["correlation_id"]
assert start_cid == end_cid, f"correlation_id mismatch: {start_cid} vs {end_cid}"
print("\n✓ start/end share the same correlation_id")

Records written: 2
  event=tool_start    correlation_id=2d07fad3-ec8f-41b4-a244-b4e1c56912fc
  event=tool_end      correlation_id=2d07fad3-ec8f-41b4-a244-b4e1c56912fc

✓ start/end share the same correlation_id


## Step 4 — Build the LangChain Agent

`AgentExecutor` wraps the three tools + your local `claude` CLI session as the backbone —
no `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` needed, it reuses whatever OAuth session
`claude auth login` already set up on this machine (see `agent/claude_code_llm.py`).

Because the CLI is a text-completion channel rather than an OpenAI-style function-calling
API, this uses the classic **ReAct** (Thought/Action/Action Input) agent pattern instead of
`create_openai_tools_agent`.  
`verbose=True` lets you see the reasoning in the cell output.

In [ ]:
%%writefile agent/build_agent.py
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from agent.tools_wrapped import fetch_tool, extract_tool, compare_tool
from agent.claude_code_llm import ClaudeCodeLLM

# Uses your existing `claude` CLI OAuth session — no API key needed.
# Pass model="opus" instead of "sonnet" if you want the bigger model.
llm = ClaudeCodeLLM(model="sonnet")

REACT_PROMPT = PromptTemplate.from_template(
    "You compare prices for a product across two approved sites. "
    "Fetch each site, extract the price, then compare. "
    "Only use the two approved domains you are given.\n\n"
    "You have access to the following tools:\n\n"
    "{tools}\n\n"
    "Use the following format:\n\n"
    "Question: the input question you must answer\n"
    "Thought: you should always think about what to do\n"
    "Action: the action to take, should be one of [{tool_names}]\n"
    "Action Input: the input to the action\n"
    "Observation: the result of the action\n"
    "... (this Thought/Action/Action Input/Observation can repeat N times)\n"
    "Thought: I now know the final answer\n"
    "Final Answer: the final answer as raw JSON with keys "
    "cheaper_site, price_a, price_b, difference (no markdown fences, no prose)\n\n"
    "Begin!\n\n"
    "Question: {input}\n"
    "Thought:{agent_scratchpad}"
)

tools = [fetch_tool, extract_tool, compare_tool]
agent = create_react_agent(llm, tools, REACT_PROMPT)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

### 4b — Import check (no API call yet)

In [3]:
import sys

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
print(f"AgentExecutor ready — tools: {[t.name for t in agent_executor.tools]}")

AgentExecutor ready — tools: ['fetch_html', 'extract_price', 'compare_prices']


## Step 5 — Run A: Unmonitored Baseline

**Before running:** make sure `claude auth status` shows `loggedIn: true` (no `OPENAI_API_KEY` needed anymore) and the two `APPROVED_DOMAINS` in `agent/tools.py` are real URLs that return HTML with a single `$XX.XX` price.

Each agent step now shells out to the `claude` CLI, so this run will be noticeably slower than an API-based run — that's expected.

This run produces `logs/execution_trace.jsonl` — your diff baseline with no governance overhead.

In [1]:
import sys, uuid
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

# Make sure the dirs from Step 0 exist even if this cell is run standalone
for d in ["agent", "runs", "analysis", "policy", "logs"]:
    Path(d).mkdir(exist_ok=True)

# Clear previous run log so we start fresh
Path("logs/execution_trace.jsonl").unlink(missing_ok=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
from agent.callback_handler import ExecutionTraceHandler

# ⚠  Replace these with your real product-page URLs before running
SITE_A_URL = "http://site-a.example.com/product"
SITE_B_URL = "http://site-b.example.com/product"
PRODUCT    = "Widget Pro 3000"

run_id_a = str(uuid.uuid4())
handler_a = ExecutionTraceHandler(run_id=run_id_a)

result_a = agent_executor.invoke(
    {"input": f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}"},
    config={"callbacks": [handler_a]},
)

print("\n=== Run A Result ===")
print(result_a)
print(f"\nRun ID : {run_id_a}")
print("Trace  : logs/execution_trace.jsonl")



> Entering new AgentExecutor chain...
I need to fetch the HTML from both approved sites first.

Action: fetch_html
Action Input: "http://site-a.example.com/product"

ConnectionError: HTTPConnectionPool(host='site-a.example.com', port=80): Max retries exceeded with url: /product (Caused by NameResolutionError("HTTPConnection(host='site-a.example.com', port=80): Failed to resolve 'site-a.example.com' ([Errno 8] nodename nor servname provided, or not known)"))

### 5b — Inspect the execution trace

In [ ]:
import pandas as pd
from pathlib import Path

trace_path = Path("logs/execution_trace.jsonl")
if trace_path.exists():
    df_exec = pd.DataFrame(
        [json.loads(l) for l in trace_path.read_text().splitlines() if l.strip()]
    )
    print(f"Total records: {len(df_exec)}")
    print(df_exec[["event", "tool", "correlation_id"]].to_string())
else:
    print("No trace yet — run Step 5 first")

## Step 6 — AgentTrust Policy

Write the YAML policy file that governs the agent's outputs.  
The embedded gateway picks this up automatically from the working directory.

In [ ]:
policy_yaml = """\
meta:
  id: price-agent
  version: "1.0"
  description: "Rules for the LangChain price-comparison agent"
  patterns:
    - "price-comparison-agent"

rules:
  - id: output_cheaper_site_present
    description: "Output must include a cheaper_site field"
    severity: high
    target: output.cheaper_site
    op: exists
    effect: deny

  - id: output_price_a_present
    description: "Output must include price_a"
    severity: high
    target: output.price_a
    op: exists
    effect: deny

  - id: output_price_b_present
    description: "Output must include price_b"
    severity: high
    target: output.price_b
    op: exists
    effect: deny

  - id: approved_domains_only
    description: "Only approved domains may be fetched"
    severity: critical
    target: output.domains_used
    op: not_in_list
    value: ["evil.com", "shadow-site.net"]
    effect: deny

  - id: large_price_difference_review
    description: "Differences above $50 are escalated for human review"
    severity: medium
    target: output.difference
    op: lte
    value: 50.0
    effect: review
"""

from pathlib import Path
policy_path = Path("policy/price_agent.yaml")
policy_path.write_text(policy_yaml)
print(f"✓ Policy written to {policy_path}")

## Step 7 — Run B: AgentTrust-Wrapped

Three integration patterns are available. This notebook uses **Pattern C** (direct `client.validate()`) for per-step governance with full access to the `ValidateResponse`.

**Requires:** AgentTrust SDK installed (Step 0) and `AGENTRUST_KEY` set in `.env`.

In [ ]:
import sys, uuid, json, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

Path("logs").mkdir(exist_ok=True)
Path("logs/governance_trace.jsonl").unlink(missing_ok=True)

# --- AgentTrust setup ---
from agentrust_sdk import embed_gateway, AgentTrustClient, BlockedError

gw     = embed_gateway()        # starts in-process SQLite gateway on :8765
client = AgentTrustClient()     # reads AGENTRUST_KEY + AGENTRUST_GATEWAY_URL from env

for mod in list(sys.modules.keys()):
    if mod.startswith("agent"):
        del sys.modules[mod]

from agent.build_agent import agent_executor
from agent.callback_handler import ExecutionTraceHandler

AGENT_ID = "price-comparison-agent"
USER     = "qa-runner"
run_id_b = str(uuid.uuid4())
handler_b = ExecutionTraceHandler(run_id=run_id_b)

# ── Governance hook: called after every tool_end ──────────────────────────────
_orig_on_tool_end = handler_b.on_tool_end

def _governed_on_tool_end(output, **kwargs):
    tool_name = kwargs.get("name", "unknown")
    serial = handler_b._call_counter.get(tool_name, 1) - 1
    cid    = handler_b._pending.get(serial, str(uuid.uuid4()))

    try:
        tool_output = json.loads(output) if isinstance(output, str) else output
    except Exception:
        tool_output = {"raw": str(output)}

    r = client.validate(
        agent_id=AGENT_ID,
        user=USER,
        input=f"tool_call:{tool_name}",
        output=tool_output,
        framework="LangChain",
        metadata={"correlation_id": cid, "run_id": run_id_b},
    )

    with open("logs/governance_trace.jsonl", "a") as f:
        f.write(json.dumps({
            "event":            "tool_governance",
            "tool":             tool_name,
            "correlation_id":   cid,
            "run_id":           run_id_b,
            "timestamp":        time.time(),
            "envelope_id":      r.envelope_id,
            "outcome":          r.decision.outcome,
            "decision_reason":  r.decision.reason,
            "policy_version":   r.decision.policy_version,
            "risk_tier":        r.risk.tier,
            "risk_score":       r.risk.score,
            "risk_reason":      r.risk.reason,
            "policy_score":     r.validation.policy_score,
            "final_confidence": r.validation.final_confidence,
            "failures":         r.validation.failures,
        }) + "\n")

    if r.blocked:
        raise BlockedError(reason=r.decision.reason, envelope_id=r.envelope_id)

    _orig_on_tool_end(output, **kwargs)

handler_b.on_tool_end = _governed_on_tool_end
# ─────────────────────────────────────────────────────────────────────────────

try:
    result_b = agent_executor.invoke(
        {"input": f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}"},
        config={"callbacks": [handler_b]},
    )
except BlockedError as e:
    print(f"BLOCKED mid-execution: {e.reason}  (envelope_id={e.envelope_id})")
    result_b = None

# ── Final-output governance ───────────────────────────────────────────────────
if result_b:
    final_output = result_b.get("output", result_b)
    if isinstance(final_output, str):
        try:
            final_output = json.loads(final_output)
        except Exception:
            final_output = {"answer": final_output}

    r_final = client.validate(
        agent_id=AGENT_ID,
        user=USER,
        input=f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}",
        output=final_output,
        framework="LangChain",
        metadata={"step": "final_output", "run_id": run_id_b},
    )

    with open("logs/governance_trace.jsonl", "a") as f:
        f.write(json.dumps({
            "event":            "final_governance",
            "correlation_id":   "final",
            "run_id":           run_id_b,
            "timestamp":        time.time(),
            "envelope_id":      r_final.envelope_id,
            "outcome":          r_final.decision.outcome,
            "decision_reason":  r_final.decision.reason,
            "risk_tier":        r_final.risk.tier,
            "policy_score":     r_final.validation.policy_score,
            "final_confidence": r_final.validation.final_confidence,
            "failures":         r_final.validation.failures,
        }) + "\n")

    if r_final.blocked:
        print(f"BLOCKED at final output: {r_final.decision.reason}")
    elif r_final.needs_review:
        print(f"ESCALATED for human review: {r_final.decision.reason}")
    else:
        print(f"APPROVED — confidence {r_final.validation.final_confidence:.1f}%")
        print(result_b)

gw.stop()
print(f"\nRun ID : {run_id_b}")
print("Governance trace → logs/governance_trace.jsonl")

### 7b — Alternative: Pattern A (`@harness` decorator)

Use this instead of the cell above if you only need **final-output** governance (no per-tool visibility).

In [ ]:
# Pattern A — decorator (simpler, final-output only)
# Uncomment to use instead of Pattern C above.

# from agentrust_sdk import harness, embed_gateway, BlockedError
# embed_gateway()  # must run before @harness is evaluated
#
# @harness(
#     agent_id="price-comparison-agent",
#     block_on_block=True,
#     block_on_review=False,
# )
# def run_price_agent(user: str, input: str) -> dict:
#     result = agent_executor.invoke({"input": input})
#     return result.get("output", {"raw": str(result)})
#
# try:
#     output = run_price_agent(
#         user="qa-runner",
#         input=f"Compare the price of {PRODUCT} on {SITE_A_URL} and {SITE_B_URL}",
#     )
#     print(output)
# except BlockedError as e:
#     print(f"BLOCKED: {e.reason}  (envelope_id={e.envelope_id})")

print("Pattern A cell — uncomment to run")

## Step 8 — Join and Compare Traces

Join the two JSONL logs on `correlation_id`.  
Run A rows have no governance columns; Run B rows carry `outcome`, `risk_tier`, `policy_score`, and `failures`.

In [ ]:
import pandas as pd, json
from pathlib import Path

def load_jsonl(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        return pd.DataFrame()
    return pd.DataFrame([json.loads(l) for l in p.read_text().splitlines() if l.strip()])

exec_trace = load_jsonl("logs/execution_trace.jsonl")
gov_trace  = load_jsonl("logs/governance_trace.jsonl")

if exec_trace.empty:
    print("No execution trace — run Step 5 first")
elif gov_trace.empty:
    print("No governance trace — run Step 7 first")
else:
    merged = exec_trace.merge(gov_trace, on="correlation_id", how="left", suffixes=("_exec", "_gov"))
    Path("analysis").mkdir(exist_ok=True)
    merged.to_csv("analysis/merged_trace.csv", index=False)

    print("=== Governance Outcome Summary ===")
    if "outcome" in merged.columns:
        print(merged.groupby("outcome")["correlation_id"].count().rename("calls"))
    else:
        print("outcome column missing — governance trace may be empty")

    print("\n=== Blocked or Escalated Steps ===")
    if "outcome" in merged.columns:
        flagged = merged[merged["outcome"].isin(["block", "escalate"])]
        cols = [c for c in ["tool_exec", "outcome", "decision_reason", "risk_tier", "policy_score", "failures"] if c in flagged.columns]
        print(flagged[cols].to_string() if not flagged.empty else "None")

    print("\n=== Risk Tier Distribution ===")
    if "risk_tier" in merged.columns and "policy_score" in merged.columns:
        print(merged.groupby("risk_tier")["policy_score"].agg(["count", "mean"]))

    print("\n=== Policy Score Stats ===")
    if "policy_score" in merged.columns:
        print(merged["policy_score"].describe())

    print("\nMerged CSV → analysis/merged_trace.csv")

### 8b — What governance caught that Run A missed

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("analysis/merged_trace.csv")
if not csv_path.exists():
    print("Run Step 8 first to generate the merged CSV")
else:
    df = pd.read_csv(csv_path)

    # Rows that were flagged (blocked or escalated)
    flagged_mask = df["outcome"].isin(["block", "escalate"]) if "outcome" in df.columns else pd.Series(False, index=df.index)
    flagged = df[flagged_mask]

    if flagged.empty:
        print("No blocks or escalations — all tool outputs passed governance in Run B")
    else:
        print(f"Governance caught {len(flagged)} flagged step(s) that Run A did not:")
        for _, row in flagged.iterrows():
            print(f"\n  tool          : {row.get('tool_exec', row.get('tool', '?'))}")
            print(f"  outcome       : {row.get('outcome')}")
            print(f"  reason        : {row.get('decision_reason')}")
            print(f"  risk_tier     : {row.get('risk_tier')}")
            print(f"  policy_score  : {row.get('policy_score')}")
            print(f"  failures      : {row.get('failures')}")